In [36]:
import os
import zipfile
import shutil
from pathlib import Path
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size
from cnnClassifier.entity import DataIngestionConfig

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self) -> str:
        '''
        Copy local dataset file to artifacts directory
        (Replaces Google Drive download for local machine usage)
        '''
        try:
            source_path = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            
            # Create artifacts directory
            os.makedirs(os.path.dirname(zip_download_dir), exist_ok=True)
            
            # Check if source is a local file path
            if source_path == "local":
                # local_data_file itself is the source
                if not os.path.exists(zip_download_dir):
                    raise FileNotFoundError(
                        f"Local file not found: {zip_download_dir}\n"
                        f"Please check your config.yaml path."
                    )
                logger.info(f"Using local file: {zip_download_dir}")
                return str(zip_download_dir)
            
            # If source_URL is an actual file path (e.g., C:/Users/.../kidney.zip)
            elif os.path.exists(source_path):
                logger.info(f"Copying data from {source_path} to {zip_download_dir}")
                
                if not os.path.exists(zip_download_dir):
                    shutil.copy(source_path, zip_download_dir)
                    logger.info(f"Copied successfully! Size: {get_size(Path(zip_download_dir))}")
                else:
                    logger.info(f"File already exists at: {zip_download_dir}")
                
                return str(zip_download_dir)
            
            else:
                raise FileNotFoundError(
                    f"Source not found: {source_path}\n"
                    f"Please set source_URL to 'local' or a valid file path in config.yaml"
                )

        except Exception as e:
            logger.error(f"Error in download_file: {e}")
            raise e

    def extract_zip_file(self):
        """
        Extracts the zip file into the data directory
        """
        try:
            unzip_path = self.config.unzip_dir
            os.makedirs(unzip_path, exist_ok=True)
            
            # Validate zip file exists
            if not os.path.exists(self.config.local_data_file):
                raise FileNotFoundError(
                    f"ZIP file not found: {self.config.local_data_file}\n"
                    f"Run download_file() first."
                )
            
            # Validate it's a valid zip
            if not zipfile.is_zipfile(self.config.local_data_file):
                raise zipfile.BadZipFile(
                    f"File is not a valid ZIP: {self.config.local_data_file}\n"
                    f"File size: {os.path.getsize(self.config.local_data_file)} bytes"
                )
            
            logger.info(f"Extracting {self.config.local_data_file} to {unzip_path}")
            
            with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
                zip_ref.extractall(unzip_path)
            
            logger.info(f"Extraction completed successfully!")
            
        except Exception as e:
            logger.error(f"Error in extract_zip_file: {e}")
            raise e

In [40]:
from cnnClassifier.config.configuration import ConfigurationManager
from cnnClassifier.components.data_ingestion import DataIngestion
from cnnClassifier import logger

try:
    # Step 1: Load configuration
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    
    # Step 2: Initialize data ingestion
    data_ingestion = DataIngestion(config=data_ingestion_config)
    
    # Step 3: Copy local file to artifacts
    data_ingestion.download_file()
    
    # Step 4: Extract ZIP
    data_ingestion.extract_zip_file()
    
    logger.info("✅ Data ingestion pipeline completed successfully!")
    
except Exception as e:
    logger.error(f"❌ Error in data ingestion pipeline: {e}")
    raise e

ModuleNotFoundError: No module named 'cnnClassifier.components.data_ingestion'

In [41]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier.entity import DataIngestionConfig  # Change this line

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [43]:
import os
from pathlib import Path

def verify_extraction(data_ingestion_config):
    unzip_dir = Path(data_ingestion_config.unzip_dir)
    
    if not unzip_dir.exists():
        logger.warning(f"❌ Unzip directory not found: {unzip_dir}")
        return False
    
    # Count all files
    extracted_files = list(unzip_dir.rglob('*'))
    logger.info(f"✅ Total items extracted: {len(extracted_files)}")
    logger.info(f"📁 Extraction directory: {unzip_dir}")
    
    # Show top-level structure
    logger.info("📂 Directory structure:")
    for item in sorted(unzip_dir.iterdir(), key=lambda x: x.name):
        if item.is_dir():
            # Count files in each split
            num_files = len(list(item.rglob('*.*')))
            logger.info(f"  ├── {item.name}/ ({num_files} files)")
        else:
            logger.info(f"  ├── {item.name}")
    
    # Show class distribution in train folder
    train_dir = unzip_dir / 'train'
    if train_dir.exists():
        logger.info("\n📊 Training set class distribution:")
        for cls in sorted(train_dir.iterdir()):
            if cls.is_dir():
                count = len(list(cls.glob('*.*')))
                logger.info(f"  ├── {cls.name}: {count} images")
    
    return True

# Usage
verify_extraction(data_ingestion_config)

[2026-06-07 05:06:29,135: INFO: 1466697244: ✅ Total items extracted: 2610]
[2026-06-07 05:06:29,136: INFO: 1466697244: 📁 Extraction directory: artifacts\data_ingestion]
[2026-06-07 05:06:29,137: INFO: 1466697244: 📂 Directory structure:]
[2026-06-07 05:06:29,139: INFO: 1466697244:   ├── data.yaml]
[2026-06-07 05:06:29,186: INFO: 1466697244:   ├── test/ (246 files)]
[2026-06-07 05:06:29,218: INFO: 1466697244:   ├── train/ (2108 files)]
[2026-06-07 05:06:29,223: INFO: 1466697244:   ├── valid/ (246 files)]
[2026-06-07 05:06:29,241: INFO: 1466697244: 
📊 Training set class distribution:]
[2026-06-07 05:06:29,261: INFO: 1466697244:   ├── images: 1054 images]
[2026-06-07 05:06:29,269: INFO: 1466697244:   ├── labels: 1054 images]


True